# Etapa 2e — Mapas de color no lineales con el fondo corregido

**Por qué.** En `02d` la mejor combinación lineal de los diez mapas corregidos (discriminante de Fisher) no le sumó nada al verde, así que ninguna mezcla lineal de rojo, verde y azul corregidos le gana. Quedan las transformaciones **no lineales** del color, que esa combinación no cubre: cocientes (verde / rojo, verde / azul), cromaticidades (un canal dividido por la suma de los tres) y saturación (`src/color.py`, `nonlinear_maps`).

**Qué se compara.** Esos cinco mapas y el verde, con el mismo preprocesamiento que en `02d` (suavizado de bloques, Wiener contra el ruido y, para cada mapa, el mejor entre corrección del fondo y ecualización adaptativa). Después, el mejor mapa combinado con cada uno de los otros y con todos (Fisher, validación cruzada por personas). Muestra fija de evaluación e imágenes en paralelo.

**Criterios:** los mismos de `02d`.


In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.color import NONLINEAR_NAMES
MAP_NAMES = ["Verde"] + NONLINEAR_NAMES
from src.data import dev_ids, load_split
from src.split import person_folds
from src.stats import fisher_direction, percentile_interval, person_bootstrap_means

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
PREPS = ["fondo", "ecualización"]
N_TRAIN = 20_000   # píxeles de vaso y de fondo por imagen para ajustar combinaciones

In [ ]:
def corrected_maps(image_id, preps=None):
    """Los diez mapas a la mitad de resolución, con bloques + Wiener contra el ruido + mejoramiento.

    Devuelve {(mapa, mejoramiento): mapa estandarizado dentro del campo detectado}.
    `preps` = {mapa: mejoramiento} limita a uno por mapa.
    """
    import numpy as np
    from PIL import Image
    from src import config as C
    from src.color import nonlinear_maps, standardize
    from src.data import load_image
    from src.restoration import background_correct, clahe, deblock, wiener_denoise
    from src.fov import detect_fov_robust, fov_diameter

    rgb = load_image(image_id)
    small = np.asarray(Image.fromarray(deblock(rgb)).reduce(C.STAGE3_FACTOR))
    region = detect_fov_robust(small)
    size = int(round(C.BACKGROUND_WINDOW_REL * fov_diameter(region))) | 1
    out = {}
    maps = {"Verde": small[..., 1].astype(np.float32) / 255.0, **nonlinear_maps(small)}
    for name, m in maps.items():
        m = wiener_denoise(m, region, C.WIENER_SIZE)
        for prep in (["fondo", "ecualización"] if preps is None else [preps[name]]):
            x = background_correct(m, region, size) if prep == "fondo" else clahe(m, region, C.CLAHE_CLIP, C.CLAHE_KERNEL_FRAC)
            out[(name, prep)] = standardize(x, region)
    return out

C_PREPS = PREPS

def pass1(image_id):
    import numpy as np
    from src.evaluation import load_sample, sample_auc, sample_values
    t0 = time.time()
    s = load_sample(image_id)
    maps = corrected_maps(image_id)
    rng = np.random.default_rng(C.SEED + len(image_id) + sum(map(ord, image_id)))
    pos, neg = np.flatnonzero(s.truth), np.flatnonzero(~s.truth)
    pick = np.r_[rng.choice(pos, N_TRAIN, replace=False), rng.choice(neg, N_TRAIN, replace=False)]
    rows, feats = [], {}
    for (name, prep), m in maps.items():
        v = sample_values(m, s)
        a = sample_auc(m, s)
        rows.append({"image": image_id, "map": name, "prep": prep, "todos": a["todos"], "finos": a["finos"]})
        feats[(name, prep)] = v[pick].astype(np.float32)
    return rows, feats, s.truth[pick], time.time() - t0

t = time.time()
out1 = __import__("src.evaluation", fromlist=["map_images"]).map_images(pass1, ids, workers=4)
print(f"Pasada 1: {time.time() - t:.0f} s para 30 imágenes; {np.mean([o[3] for o in out1]):.1f} s por imagen dentro de cada proceso")
R = pd.DataFrame([r for o in out1 for r in o[0]])
R.to_csv(ROOT / "results" / "02e_color_no_lineal.csv", index=False)

## Ranking

In [ ]:
def oriented_table(R, prep_choice=None, measure="todos"):
    w = R.pivot_table(index="image", columns=["map", "prep"], values=measure)
    flip = w.mean() < 0.5
    w.loc[:, flip] = 1 - w.loc[:, flip]
    return w

W = oriented_table(R)
best_prep = {m: W[m].mean().idxmax() for m in MAP_NAMES}
Wb = pd.DataFrame({m: W[(m, best_prep[m])] for m in MAP_NAMES})
Fb = oriented_table(R, measure="finos")
Fb = pd.DataFrame({m: Fb[(m, best_prep[m])] for m in MAP_NAMES})
best = Wb.mean().idxmax()
diffs = Wb.sub(Wb[best], axis=0)
lo, hi = percentile_interval(person_bootstrap_means(diffs, dev["person"]))
rank = pd.DataFrame({"mejoramiento": pd.Series(best_prep), "área": Wb.mean(), "área finos": Fb.mean(),
                     "peor imagen": Wb.min(), "vs mejor": diffs.mean(), "límite inferior": lo,
                     "no inferior": lo >= -C.NONINF_MARGIN_AUC}).sort_values("área", ascending=False)
rank.round(4)

In [ ]:
picks = []
for k in range(5):
    tr = folds.index[folds != k]
    picks.append(Wb.loc[tr].mean().idxmax())
print("Mejor mapa por vuelta:", picks)

## Combinaciones con el mejor mapa (pasada 2)

In [ ]:
prep_of = best_prep
feat_cols = MAP_NAMES
X_by_img = {o[0][0]["image"]: np.stack([o[1][(m, prep_of[m])] for m in feat_cols], axis=1) for o in out1}
y_by_img = {o[0][0]["image"]: o[2] for o in out1}
ib = feat_cols.index(best)
best_sign = 1 if R[(R["map"] == best) & (R["prep"] == prep_of[best])]["todos"].mean() >= 0.5 else -1
COMBOS = {f"{best} + {m}": [ib, feat_cols.index(m)] for m in feat_cols if m != best}
COMBOS["todos"] = list(range(len(feat_cols)))
weights = {}
for k in range(5):
    tr = folds.index[folds != k]
    X = np.concatenate([X_by_img[i] for i in tr]); y = np.concatenate([y_by_img[i] for i in tr])
    for combo, cols in COMBOS.items():
        weights[(k, combo)] = fisher_direction(X[:, cols], y)

def pass2(image_id):
    from src.evaluation import load_sample, sample_values
    from src.metrics import auc_by_subset
    s = load_sample(image_id)
    maps = corrected_maps(image_id, prep_of)
    V = np.stack([sample_values(maps[(m, prep_of[m])], s) for m in feat_cols], axis=1)
    k = int(folds[image_id])
    row = {"image": image_id, f"{best} solo": auc_by_subset(best_sign * V[:, ib], s.truth, s.subsets())["todos"]}
    for combo, cols in COMBOS.items():
        row[combo] = auc_by_subset(V[:, cols] @ weights[(k, combo)], s.truth, s.subsets())["todos"]
    return row

t = time.time()
Cmb = pd.DataFrame(__import__("src.evaluation", fromlist=["map_images"]).map_images(pass2, ids, workers=4)).set_index("image")
print(f"Pasada 2: {time.time() - t:.0f} s")
Cmb.to_csv(ROOT / "results" / "02e_combinaciones.csv")
gain = Cmb.drop(columns=f"{best} solo").sub(Cmb[f"{best} solo"], axis=0)
lo, hi = percentile_interval(person_bootstrap_means(gain, dev["person"]))
comp = pd.DataFrame({"área combinada": Cmb.drop(columns=f"{best} solo").mean(), "ganancia": gain.mean(),
                     "límite inferior": lo, "límite superior": hi}).sort_values("ganancia", ascending=False)
print(f"{best} solo, personas de afuera: {Cmb[f'{best} solo'].mean():.4f}")
comp.round(4)